# Brain extraction across modalities and T1 variants

Choose one of the 11 modality strings supported by this ANTsTorch checkout.
This notebook runs **one selected modality**, not all models in a loop.
The default `t1threetissue` uses the downloadable sample T1. FLAIR and MRA use matching tutorial downloads by default; other acquisition
types require a matching local image; a T1 is not substituted for a different
contrast. **Resources:** templates and pretrained weights download on first
use and are cached under `~/.antstorch`. These 3-D workflows can require
substantial memory and several minutes or longer on CPU.

| `modality` | Required image | Function return |
| --- | --- | --- |
| `t1` | Scalar 3-D T1 MRI | Brain probability image |
| `t2` | Scalar 3-D T2 MRI | Brain probability image |
| `t2star` | Scalar 3-D T2* MRI | Brain probability image |
| `flair` | Scalar 3-D FLAIR MRI | Brain probability image |
| `bold` | Scalar 3-D BOLD reference or temporal-mean volume | Brain probability image |
| `fa` | Scalar 3-D fractional-anisotropy image | Brain probability image |
| `pet` | Scalar 3-D PET image | Brain probability image |
| `mra` | Scalar 3-D MR angiography image | Brain probability image |
| `t1threetissue` | Scalar 3-D T1 MRI | Dictionary: labels 0–3 and four probability images |
| `t1hemi` | Scalar 3-D T1 MRI | Dictionary: background and two hemisphere labels |
| `t1lobes` | Scalar 3-D T1 MRI | Dictionary: background and five region labels |

The last three are T1 processing variants rather than different acquisitions.
The implementation applies `t1threetissue` internally before `t1hemi` and
`t1lobes`, so those variants load more than one network.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import os
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = os.environ.get("ANTSTORCH_EXAMPLE_DEVICE", "cpu")
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

## Select the modality and its matching input
Set the two variables below, or use `ANTSTORCH_EXAMPLE_BRAIN_MODALITY` and
`ANTSTORCH_EXAMPLE_BRAIN_IMAGE`. Use an absolute path for a local image.
For a 4-D fMRI series, select a suitable 3-D reference or compute a temporal
mean first; do not pass the 4-D series directly to `brain_extraction`.

In [ ]:
import os
from pathlib import Path

modality = os.environ.get("ANTSTORCH_EXAMPLE_BRAIN_MODALITY", "t1threetissue")
image_path = os.environ.get("ANTSTORCH_EXAMPLE_BRAIN_IMAGE")
# Alternatively: modality = "flair"; image_path = "/absolute/path/flair.nii.gz"
supported_modalities = ("t1", "t2", "t2star", "flair", "bold", "fa", "pet", "mra",
                        "t1threetissue", "t1hemi", "t1lobes")
t1_variants = {"t1", "t1threetissue", "t1hemi", "t1lobes"}
if modality not in supported_modalities:
    raise ValueError(f"Choose one of {supported_modalities}; got {modality!r}.")
if image_path is None:
    tutorial_inputs = {"flair": "tutorialWmhFlair", "mra": "tutorialMra"}
    if modality in t1_variants:
        image_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
    elif modality in tutorial_inputs:
        image_path = antstorch.get_antstorch_data(tutorial_inputs[modality])
    else:
        raise FileNotFoundError(f"Supply a matching 3-D {modality} image via image_path or ANTSTORCH_EXAMPLE_BRAIN_IMAGE.")
if not Path(image_path).is_file():
    raise FileNotFoundError(f"Image does not exist: {image_path}")
image = ants.image_read(str(image_path)).clone("float")
if image.dimension != 3 or image.has_components:
    raise ValueError("Expected a scalar 3-D image; reduce a 4-D BOLD series to a 3-D reference first.")
assert np.isfinite(image.numpy()).all() and np.ptp(image.numpy()) > 0
print("Modality:", modality, "input:", image_path)
print("Shape:", image.shape, "spacing:", image.spacing)

## Run the selected model
PET uses its own template resampling and crop/pad inside the function. All
supported modalities have weight download URLs in the current catalog.
Model availability is not evidence of accuracy on every scanner or acquisition.

In [ ]:
with time_step('Brain extraction (result)'):
    result = antstorch.brain_extraction(image, modality=modality, device=device, verbose=True)

## Convert the result into the appropriate mask
For binary models, threshold the returned brain probability image at 0.5.
For `t1threetissue`, use **label 1**, the brain label used by ANTsTorch's own
hemisphere/lobe preprocessing; including every nonzero label would retain
other classes. For `t1hemi` and `t1lobes`, the foreground region mask is the
union of their nonbackground labels. The latter is a region-coverage mask,
not a replacement for a whole-brain extraction mask.

In [ ]:
def validate_geometry(output, reference):
    assert output.shape == reference.shape
    assert np.isfinite(output.numpy()).all()
    for attribute in ("spacing", "origin", "direction"):
        np.testing.assert_allclose(getattr(output, attribute), getattr(reference, attribute))

segmentation = None
if modality in {"t1threetissue", "t1hemi", "t1lobes"}:
    segmentation = result["segmentation_image"]
    probabilities = result["probability_images"]
    class_count = {"t1threetissue": 4, "t1hemi": 3, "t1lobes": 6}[modality]
    assert len(probabilities) == class_count
    validate_geometry(segmentation, image)
    assert np.isin(segmentation.numpy(), np.arange(class_count)).all()
    if modality == "t1threetissue":
        mask = ants.threshold_image(segmentation, 1, 1, 1, 0)
        mask_name = "brain_mask"
    else:
        mask = ants.threshold_image(segmentation, 1, class_count-1, 1, 0)
        mask_name = "region_mask"
else:
    probabilities = [result]
    threshold = 0.5  # Demonstration value: inspect/tune on the intended data.
    mask = ants.threshold_image(result, threshold, 1., 1, 0)
    mask_name = "brain_mask"
for probability in probabilities:
    validate_geometry(probability, image)
    values = probability.numpy()
    assert values.min() >= -1e-6 and values.max() <= 1 + 1e-6
validate_geometry(mask, image)
assert np.isin(mask.numpy(), [0, 1]).all()
assert (mask.numpy() > 0).any()
extracted = image * mask

## Inspect the mask and, when available, the region labels
Displays use masks or discrete labels, not probability overlays. Inspect the
boundaries and left/right orientation. Region coverage and whole-brain masks
answer different questions; compare the labels before interpreting volumes.

In [ ]:
ants.plot(image, overlay=mask, axis=2, nslices=6,
          overlay_cmap="autumn", overlay_alpha=0.5)
if segmentation is not None:
    ants.plot(image, overlay=segmentation, axis=2, nslices=6,
              overlay_cmap="jet", overlay_alpha=0.5)
output_dir = Path("example_outputs/brain_extraction_modalities") / modality
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(mask, str(output_dir / f"{mask_name}.nii.gz"))
ants.image_write(extracted, str(output_dir / "masked_image.nii.gz"))
if segmentation is not None:
    ants.image_write(segmentation, str(output_dir / "labels.nii.gz"))
    for index, probability in enumerate(probabilities):
        ants.image_write(probability, str(output_dir / f"probability_{index}.nii.gz"))
else:
    ants.image_write(probabilities[0], str(output_dir / "brain_probability.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

Changing `modality` requires rerunning from the input-selection cell with a
matching image. ANTsTorch caches the loaded networks and templates by modality,
channel count and device. After experimenting with several models, release
their cached references with `antstorch.brain_extraction.clear_cache()` when
those resources are no longer needed. This notebook does not test numerical
agreement with ANTsPyNet or anatomical accuracy for the different contrasts.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")